# Atom id extraction for `Boltz2` colvars
This notebook recreates the processed structure files from the Boltz input data. This allows users to find the atom id numbers to which they want to apply their colvars. `Boltz2` needs to be run once already on the target system so that several necessary files have been generated already.

## Generate the structures for atom extraction
#### Load the necessary routines

In [ ]:
import torch
import multiprocessing
from pathlib import Path

In [ ]:
from boltz.data.module.inferencev2 import Boltz2InferenceDataModule
from boltz.data.types import MSA, Manifest, Record

In [ ]:
from boltz.main import filter_inputs_structure, BoltzProcessedInput, process_inputs, check_inputs
from boltz.data.module.inferencev2 import load_input

## Inputs to be updated by user for their system

In [ ]:
data = "/home/sotomayor/Documents/klauscj/boltz/systems/dna.fasta"
out_dir = "/home/sotomayor/Documents/klauscj/boltz/boltz_results_dna"

## Inputs that usually stay constant

In [ ]:
cache   = "~/.boltz/boltz2"
use_msa_server = True
msa_server_url = "https://api.colabfold.com"
msa_pairing_strategy = "greedy"
msa_server_username = None
msa_server_password = None
api_key_header = None
api_key_value = None
boltz2 = True
preprocessing_threads = multiprocessing.cpu_count()
max_msa_seqs = 8192
override = False
num_workers = 2
method = None
affinity = False

#### Generate the files
Note: I omitted the `features=dataset.featurizer.process(` command that I had in Boltz1 colvars. You can find it in the `src/boltz/data/module/trainingv2.py` file.

In [ ]:
# Set cache path
cache = Path(cache).expanduser()
cache.mkdir(parents=True, exist_ok=True)

In [ ]:
# Create output directories
data = Path(data).expanduser()
out_dir = Path(out_dir).expanduser()
out_dir = out_dir / f"boltz_results_{data.stem}"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Validate inputs
data = check_inputs(data)

In [ ]:
# Process inputs
ccd_path = cache / "ccd.pkl"
mol_dir = cache / "mols"

process_inputs(
        data=data,
        out_dir=out_dir,
        ccd_path=ccd_path,
        mol_dir=mol_dir,
        use_msa_server=use_msa_server,
        msa_server_url=msa_server_url,
        msa_pairing_strategy=msa_pairing_strategy,
        msa_server_username=msa_server_username,
        msa_server_password=msa_server_password,
        api_key_header=api_key_header,
        api_key_value=api_key_value,
        boltz2=boltz2,
        preprocessing_threads=preprocessing_threads,
        max_msa_seqs=max_msa_seqs,
)

In [ ]:
# Load manifest
manifest = Manifest.load(out_dir / "processed" / "manifest.json")

# Filter out existing predictions
filtered_manifest = filter_inputs_structure(
    manifest=manifest,
    outdir=out_dir,
    override=override
)

In [ ]:
# Load processed data
processed_dir = out_dir / "processed"
processed = BoltzProcessedInput(
        manifest=filtered_manifest,
        targets_dir=processed_dir / "structures",
        msa_dir=processed_dir / "msa",
        constraints_dir=(
            (processed_dir / "constraints")
            if (processed_dir / "constraints").exists()
            else None
        ),
        template_dir=(
            (processed_dir / "templates")
            if (processed_dir / "templates").exists()
            else None
        ),
        extra_mols_dir=(
            (processed_dir / "mols") if (processed_dir / "mols").exists() else None
        ),
)

In [ ]:
# Create data module
data_module = Boltz2InferenceDataModule(
                manifest=processed.manifest,
                target_dir=processed.targets_dir,
                msa_dir=processed.msa_dir,
                mol_dir=mol_dir,
                num_workers=num_workers,
                constraints_dir=processed.constraints_dir,
                template_dir=processed.template_dir,
                extra_mols_dir=processed.extra_mols_dir,
                override_method=method,
)
dataset = data_module.predict_dataloader().dataset

In [ ]:
record = dataset.manifest.records[0]

In [ ]:
input_data = load_input(
            record=record,
            target_dir=processed.targets_dir,
            msa_dir=processed.msa_dir,
            constraints_dir=processed.constraints_dir,
            template_dir=processed.template_dir,
            extra_mols_dir=processed.extra_mols_dir,
            affinity=affinity,
)

In [ ]:
tokenized = dataset.tokenizer.tokenize(input_data)

#### Processing the atomic coordinates

In [ ]:
input_data.structure.residues[0:12]

In [ ]:
input_data.structure.residues

# Export a connectivity list
We export an index list which connects the C-terminal backbone atom within a residue to the N-initial backbone atom within the next residue. Because the polypeptide bond runs along a single chain, the user must enter the starting residue index of each chain.

According to `src/boltz/data/const.py`, the dna index atom lists are ordered like

DA: ["P", "OP1", "OP2", "O5'", "C5'", "C4'", "O4'", "C3'", "O3'", "C2'", "C1'", "N9", "C8", "N7", "C5", "C6", "N6", "N1", "C2", "N3", "C4"]

DG: ["P", "OP1", "OP2", "O5'", "C5'", "C4'", "O4'", "C3'", "O3'", "C2'", "C1'", "N9", "C8", "N7", "C5", "C6", "O6", "N1", "C2", "N2", "N3", "C4"]

DC: ["P", "OP1", "OP2", "O5'", "C5'", "C4'", "O4'", "C3'", "O3'", "C2'", "C1'", "N1", "C2", "O2", "N3", "C4", "N4", "C5", "C6"]

DT: ["P", "OP1", "OP2", "O5'", "C5'", "C4'", "O4'", "C3'", "O3'", "C2'", "C1'", "N1", "C2", "O2", "N3", "C4", "O4", "C5", "C7", "C6"]

We need to link the O3 of the first base to the phosphate of the next base.
## Inputs to be specified by user

In [ ]:
# User enter list of each chain's first residue
firstresid_chains = [0,12]

## Generate the per chain connectivity lists along the dna backbone

In [ ]:
nchains = len(firstresid_chains)
firstresid_chains.append(len(input_data.structure.residues))

In [ ]:
connected_atom_index = []
for k in range(nchains):
    connected_atom_list_O3 = torch.tensor([input_data.structure.residues[idend]["atom_idx"]+8 
                                          for idend in range(firstresid_chains[k],firstresid_chains[k+1]-1)])
    connected_atom_list_P = torch.tensor([input_data.structure.residues[idbeg]["atom_idx"] 
                                          for idbeg in range(firstresid_chains[k]+1,firstresid_chains[k+1])])
    chain_connected_atom_list = torch.hstack((connected_atom_list_O3.reshape((-1,1)),connected_atom_list_P.reshape((-1,1))))
    chain_connected_atom_list = chain_connected_atom_list.t()
    connected_atom_index.append(chain_connected_atom_list)
connected_atom_index = torch.hstack(connected_atom_index)
connected_atom_index = connected_atom_index.unsqueeze(0)

In [ ]:
torch.save(connected_atom_index,"../../connected_atom_index.pt")